# Deep Learning Mini Project

Classification of waldo my guy thanks claude :)

In [4]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image, ImageDraw
import numpy as np
import os
import glob

In [5]:
class WaldoPatchDataset(Dataset):
    """
    Expects folder structure:
        patches/
            waldo/        ← patches containing Waldo
            not_waldo/    ← patches without Waldo
    """
 
    def __init__(self, root_dir, patch_size=128, augment=True):
        self.patch_size = patch_size
 
        self.waldo_paths = glob.glob(os.path.join(root_dir, "waldo", "*"))
        self.not_waldo_paths = glob.glob(os.path.join(root_dir, "not_waldo", "*"))
 
        self.all_paths = self.waldo_paths + self.not_waldo_paths
        self.labels = [1] * len(self.waldo_paths) + [0] * len(self.not_waldo_paths)
 
        print(f"Loaded {len(self.waldo_paths)} positive, "
              f"{len(self.not_waldo_paths)} negative patches")
 
        # Augmentation for training
        if augment:
            self.transform = transforms.Compose([
                transforms.Resize((patch_size, patch_size)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
                transforms.RandomRotation(15),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225]),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((patch_size, patch_size)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225]),
            ])
 
    def __len__(self):
        return len(self.all_paths)
 
    def __getitem__(self, idx):
        img = Image.open(self.all_paths[idx]).convert("RGB")
        img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label
 
    def get_sampler_weights(self):
        """Returns weights for WeightedRandomSampler to handle class imbalance."""
        counts = [len(self.not_waldo_paths), len(self.waldo_paths)]
        class_weights = [1.0 / c for c in counts]
        sample_weights = [class_weights[label] for label in self.labels]
        return sample_weights

In [ ]:
class WaldoPatchClassifier(nn.Module):
    """ResNet-18 backbone → binary classification (Waldo or not)."""
 
    def __init__(self, pretrained=True):
        super().__init__()
        backbone = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT if pretrained else None
        )
        num_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.features = backbone
 
        self.classifier = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1),
        )
 
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
def train(model, train_loader, val_loader, epochs=25, lr=1e-4, device="cuda"):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.5)
    loss_fn = nn.BCEWithLogitsLoss()
 
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
 
    for epoch in range(epochs):
        # ── Train ──
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs).squeeze(1)
            loss = loss_fn(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
 
        train_loss = running_loss / len(train_loader.dataset)
 
        # ── Validate ──
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs).squeeze(1)
                val_loss += loss_fn(logits, labels).item() * imgs.size(0)
                preds = (torch.sigmoid(logits) > 0.5).float()
                correct += (preds == labels).sum().item()
                total += labels.size(0)
 
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total
        scheduler.step()
 
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
 
        print(f"Epoch {epoch+1:3d}/{epochs} │ "
              f"Train Loss: {train_loss:.4f} │ "
              f"Val Loss: {val_loss:.4f} │ "
              f"Val Acc: {val_acc:.4f}")
 
    return history

In [ ]:
def sliding_window_detect(model, image_path, patch_size=128, stride=32,
                          scales=[1.0, 0.75, 0.5], device="cuda", threshold=0.8):
    """
    Run a trained patch classifier across a full image at multiple scales.
    Returns a heatmap and list of detections.
    """
    model.eval()
    original = Image.open(image_path).convert("RGB")
    orig_w, orig_h = original.size
 
    transform = transforms.Compose([
        transforms.Resize((patch_size, patch_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])
 
    # Accumulate scores across scales
    heatmap = np.zeros((orig_h, orig_w), dtype=np.float32)
    count_map = np.zeros((orig_h, orig_w), dtype=np.float32)
    detections = []
 
    for scale in scales:
        # Resize image for this scale
        scaled_w, scaled_h = int(orig_w * scale), int(orig_h * scale)
        scaled_img = original.resize((scaled_w, scaled_h), Image.BILINEAR)
 
        # Slide window
        patches, coords = [], []
        for y in range(0, scaled_h - patch_size + 1, stride):
            for x in range(0, scaled_w - patch_size + 1, stride):
                patch = scaled_img.crop((x, y, x + patch_size, y + patch_size))
                patches.append(transform(patch))
                # Map back to original image coordinates
                coords.append((
                    int(x / scale), int(y / scale),
                    int((x + patch_size) / scale), int((y + patch_size) / scale)
                ))
 
        if not patches:
            continue
 
        # Batch inference
        batch = torch.stack(patches).to(device)
        with torch.no_grad():
            scores = torch.sigmoid(model(batch).squeeze(1)).cpu().numpy()
 
        # Fill heatmap and collect detections
        for score, (x1, y1, x2, y2) in zip(scores, coords):
            x1c = max(0, x1)
            y1c = max(0, y1)
            x2c = min(orig_w, x2)
            y2c = min(orig_h, y2)
            heatmap[y1c:y2c, x1c:x2c] += score
            count_map[y1c:y2c, x1c:x2c] += 1
 
            if score > threshold:
                detections.append({"bbox": (x1, y1, x2, y2), "score": float(score)})
 
    # Average heatmap
    count_map[count_map == 0] = 1
    heatmap /= count_map
 
    return heatmap, detections

In [ ]:
def nms(detections, iou_threshold=0.3):
    """Simple NMS to remove overlapping detections."""
    if not detections:
        return []
 
    detections = sorted(detections, key=lambda d: d["score"], reverse=True)
    keep = []
 
    while detections:
        best = detections.pop(0)
        keep.append(best)
        remaining = []
        for det in detections:
            if box_iou(best["bbox"], det["bbox"]) < iou_threshold:
                remaining.append(det)
        detections = remaining
 
    return keep
 
 
def box_iou(box_a, box_b):
    """IoU between two (x1, y1, x2, y2) boxes."""
    xa = max(box_a[0], box_b[0])
    ya = max(box_a[1], box_b[1])
    xb = min(box_a[2], box_b[2])
    yb = min(box_a[3], box_b[3])
 
    inter = max(0, xb - xa) * max(0, yb - ya)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
 
    return inter / (area_a + area_b - inter + 1e-6)

In [ ]:
def visualize_detections(image_path, detections, save_path="result.png"):
    """Draw bounding boxes on the image."""
    img = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(img)
 
    for det in detections:
        x1, y1, x2, y2 = det["bbox"]
        score = det["score"]
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, y1 - 12), f"{score:.2f}", fill="red")
 
    img.save(save_path)
    print(f"Saved result to {save_path}")
    return img

In [ ]:
if __name__ == "__main__":
    PATCH_SIZE = 128
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
 
    # ── Training ──
    train_ds = WaldoPatchDataset("data/patches/train", patch_size=PATCH_SIZE, augment=True)
    val_ds = WaldoPatchDataset("data/patches/val", patch_size=PATCH_SIZE, augment=False)
 
    # Weighted sampler to handle class imbalance
    sampler = WeightedRandomSampler(
        weights=train_ds.get_sampler_weights(),
        num_samples=len(train_ds),
        replacement=True
    )
 
    train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=4)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=4)
 
    model = WaldoPatchClassifier(pretrained=True)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {total_params:,}")
 
    history = train(model, train_loader, val_loader, epochs=25, lr=1e-4, device=DEVICE)
    torch.save(model.state_dict(), "waldo_patch_classifier.pth")
 
    # ── Inference on a full image ──
    print("\n── Running sliding window detection ──")
    heatmap, detections = sliding_window_detect(
        model,
        image_path="data/original/waldo_page_01.png",
        patch_size=PATCH_SIZE,
        stride=32,
        scales=[1.0, 0.75, 0.5],
        device=DEVICE,
        threshold=0.85,
    )
 
    # Apply NMS
    final_detections = nms(detections, iou_threshold=0.3)
    print(f"Found {len(final_detections)} detection(s) after NMS")
 
    # Visualize
    visualize_detections("data/original/waldo_page_01.png", final_detections)